Імпорт пакетів

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Preprocessing
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Model
from sklearn.linear_model import LogisticRegression

# Metrics
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print("Всі пакети успішно імпортовані")

Всі пакети успішно імпортовані


Завантаження датасету

In [2]:
# Завантаження датасету Rain in Australia
# Датасет: https://www.kaggle.com/jsphyg/weather-dataset-rattle-package
# Файл: weatherAUS.csv

import pandas as pd
df = pd.read_csv('weatherAUS.csv')
print(f"✓ Датасет: {df.shape}")


✓ Датасет: (145460, 23)


3.1 Видалення ознак з великою кількістю пропущених значень

In [3]:
# Визначення порогу для видалення (більше 40% пропусків)
missing_threshold = 0.4
missing_percentage = df.isnull().sum() / len(df)

print("Відсоток пропущених значень по ознаках:")
print(missing_percentage.sort_values(ascending=False))

# Видалення ознак з великою кількістю пропусків
columns_to_drop = missing_percentage[missing_percentage > missing_threshold].index.tolist()
print(f"\nОзнаки для видалення: {columns_to_drop}")

df = df.drop(columns=columns_to_drop)
print(f"\nДатасет після видалення ознак: {df.shape}")

Відсоток пропущених значень по ознаках:
Sunshine         0.480098
Evaporation      0.431665
Cloud3pm         0.408071
Cloud9am         0.384216
Pressure9am      0.103568
Pressure3pm      0.103314
WindDir9am       0.072639
WindGustDir      0.070989
WindGustSpeed    0.070555
Humidity3pm      0.030984
WindDir3pm       0.029066
Temp3pm          0.024811
RainTomorrow     0.022460
Rainfall         0.022419
RainToday        0.022419
WindSpeed3pm     0.021050
Humidity9am      0.018246
Temp9am          0.012148
WindSpeed9am     0.012148
MinTemp          0.010209
MaxTemp          0.008669
Location         0.000000
Date             0.000000
dtype: float64

Ознаки для видалення: ['Evaporation', 'Sunshine', 'Cloud3pm']

Датасет після видалення ознак: (145460, 20)


In [4]:
# Видалення колонок Date та RainTomorrow з розгляду на цьому етапі
df_features = df.drop(columns=['Date', 'RainTomorrow'])

# Розділення на числові та категоріальні ознаки
numeric_features = df_features.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = df_features.select_dtypes(include=['object']).columns.tolist()

print("Числові ознаки:")
print(numeric_features)
print(f"\nКатегоріальні ознаки:")
print(categorical_features)

Числові ознаки:
['MinTemp', 'MaxTemp', 'Rainfall', 'WindGustSpeed', 'WindSpeed9am', 'WindSpeed3pm', 'Humidity9am', 'Humidity3pm', 'Pressure9am', 'Pressure3pm', 'Cloud9am', 'Temp9am', 'Temp3pm']

Категоріальні ознаки:
['Location', 'WindGustDir', 'WindDir9am', 'WindDir3pm', 'RainToday']


In [5]:
# Конвертація Date в datetime
df['Date'] = pd.to_datetime(df['Date'])

# Створення ознак Year та Month
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month

print("Приклад обробки часових ознак:")
print(df[['Date', 'Year', 'Month']].head())
print(f"\nРоки у датасеті: {sorted(df['Year'].unique())}")
print(f"Максимальний рік: {df['Year'].max()}")

Приклад обробки часових ознак:
        Date  Year  Month
0 2008-12-01  2008     12
1 2008-12-02  2008     12
2 2008-12-03  2008     12
3 2008-12-04  2008     12
4 2008-12-05  2008     12

Роки у датасеті: [2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017]
Максимальний рік: 2017


In [6]:
# Year обробляється як числова ознака
numeric_features.append('Year')

# Month обробляється як категоріальна ознака
categorical_features.append('Month')

print(f"Оновлені числові ознаки: {numeric_features}")
print(f"Оновлені категоріальні ознаки: {categorical_features}")

Оновлені числові ознаки: ['MinTemp', 'MaxTemp', 'Rainfall', 'WindGustSpeed', 'WindSpeed9am', 'WindSpeed3pm', 'Humidity9am', 'Humidity3pm', 'Pressure9am', 'Pressure3pm', 'Cloud9am', 'Temp9am', 'Temp3pm', 'Year']
Оновлені категоріальні ознаки: ['Location', 'WindGustDir', 'WindDir9am', 'WindDir3pm', 'RainToday', 'Month']


In [7]:
# Визначення максимального року
max_year = df['Year'].max()
min_year = df['Year'].min()

print(f"Діапазон років у датасеті: {min_year} - {max_year}")

# Булева індексація: тест = останній рік, тренування = решта років
train_mask = df['Year'] != max_year
test_mask = df['Year'] == max_year

# Розділення всього датасету
df_train = df[train_mask].copy()
df_test = df[test_mask].copy()

print(f"\nТренувальна вибірка: {df_train.shape[0]} рядків (роки: {sorted(df_train['Year'].unique())})")
print(f"Тестова вибірка: {df_test.shape[0]} рядків (рік: {sorted(df_test['Year'].unique())})")

# Видалення пропусків у цільовій змінній (RainTomorrow)
print(f"\nПропущені значення RainTomorrow до видалення:")
print(f"Тренування: {df_train['RainTomorrow'].isnull().sum()}")
print(f"Тест: {df_test['RainTomorrow'].isnull().sum()}")

df_train = df_train[df_train['RainTomorrow'].notna()]
df_test = df_test[df_test['RainTomorrow'].notna()]

print(f"\nЗагальний обсяг після видалення пропусків у RainTomorrow:")
print(f"Тренування: {df_train.shape[0]}")
print(f"Тест: {df_test.shape[0]}")

Діапазон років у датасеті: 2007 - 2017

Тренувальна вибірка: 136837 рядків (роки: [2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016])
Тестова вибірка: 8623 рядків (рік: [2017])

Пропущені значення RainTomorrow до видалення:
Тренування: 3110
Тест: 157

Загальний обсяг після видалення пропусків у RainTomorrow:
Тренування: 133727
Тест: 8466


4. Відновлення пропущених даних за допомогою SimpleImputer

In [8]:
# Розділення ознак і цільової змінної
X_train_num = df_train[numeric_features]
X_test_num  = df_test[numeric_features]

X_train_cat = df_train[categorical_features]
X_test_cat  = df_test[categorical_features]

y_train = df_train['RainTomorrow']
y_test  = df_test['RainTomorrow']

# Заповнення пропусків у числових ознаках — медіаною
num_imputer = SimpleImputer(strategy='median')
X_train_num_imputed = num_imputer.fit_transform(X_train_num)
X_test_num_imputed  = num_imputer.transform(X_test_num)

# Заповнення пропусків у категоріальних ознаках — найчастішим значенням
cat_imputer = SimpleImputer(strategy='most_frequent')
X_train_cat_imputed = cat_imputer.fit_transform(X_train_cat)
X_test_cat_imputed  = cat_imputer.transform(X_test_cat)

print("Пропущені значення заповнено")
print(f"Числові: train {X_train_num_imputed.shape}, test {X_test_num_imputed.shape}")
print(f"Категоріальні: train {X_train_cat_imputed.shape}, test {X_test_cat_imputed.shape}")

Пропущені значення заповнено
Числові: train (133727, 14), test (8466, 14)
Категоріальні: train (133727, 6), test (8466, 6)


5. Нормалізація числових ознак за допомогою StandardScaler

In [9]:
scaler = StandardScaler()
X_train_num_scaled = scaler.fit_transform(X_train_num_imputed)
X_test_num_scaled  = scaler.transform(X_test_num_imputed)

print("Числові ознаки нормалізовано")
print(f"Середнє (train, перші 3 ознаки): {X_train_num_scaled.mean(axis=0)[:3].round(6)}")
print(f"Std     (train, перші 3 ознаки): {X_train_num_scaled.std(axis=0)[:3].round(6)}")

Числові ознаки нормалізовано
Середнє (train, перші 3 ознаки): [-0.  0. -0.]
Std     (train, перші 3 ознаки): [1. 1. 1.]


6. Кодування категоріальних ознак за допомогою OneHotEncoder

In [10]:
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_train_cat_encoded = encoder.fit_transform(X_train_cat_imputed)
X_test_cat_encoded  = encoder.transform(X_test_cat_imputed)

print("Категоріальні ознаки закодовано")
print(f"Кількість ознак після OHE: {X_train_cat_encoded.shape[1]}")

Категоріальні ознаки закодовано
Кількість ознак після OHE: 111


7. Об'єднання ознак і навчання моделі LogisticRegression

In [11]:
# Об'єднання числових і категоріальних ознак
X_train_final = np.hstack([X_train_num_scaled, X_train_cat_encoded])
X_test_final  = np.hstack([X_test_num_scaled,  X_test_cat_encoded])

print(f"Розмір тренувальної матриці: {X_train_final.shape}")
print(f"Розмір тестової матриці:     {X_test_final.shape}")

# Порівняння різних solvers
for solver in ['lbfgs', 'liblinear', 'saga']:
    model = LogisticRegression(solver=solver, max_iter=1000, random_state=42)
    model.fit(X_train_final, y_train)
    acc = accuracy_score(y_test, model.predict(X_test_final))
    print(f"solver={solver:12s} -> accuracy={acc:.4f}")

Розмір тренувальної матриці: (133727, 125)
Розмір тестової матриці:     (8466, 125)


solver=lbfgs        -> accuracy=0.8485


solver=liblinear    -> accuracy=0.8482


solver=saga         -> accuracy=0.8482


8. Оцінка метрик моделі та порівняння з базовою

In [12]:
# Фінальна модель
best_model = LogisticRegression(solver='lbfgs', max_iter=1000, random_state=42)
best_model.fit(X_train_final, y_train)
y_pred = best_model.predict(X_test_final)

print("=== Classification Report ===")
print(classification_report(y_test, y_pred))

print("=== Confusion Matrix ===")
print(confusion_matrix(y_test, y_pred))

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")

=== Classification Report ===
              precision    recall  f1-score   support

          No       0.87      0.95      0.91      6703
         Yes       0.72      0.45      0.55      1763

    accuracy                           0.85      8466
   macro avg       0.79      0.70      0.73      8466
weighted avg       0.84      0.85      0.83      8466

=== Confusion Matrix ===
[[6392  311]
 [ 972  791]]
Accuracy: 0.8485


In [13]:
print("""
Висновки:
1. Розбиття на тренувальну/тестову вибірки за часовим принципом (останній рік —
   тестова) дозволяє змоделювати реальні умови: навчання на "минулому",
   оцінка на умовному "майбутньому". Це запобігає надмірному оптимізму в метриках.

2. Трансформери (SimpleImputer, StandardScaler, OneHotEncoder) навчаються лише на
   тренувальних даних (fit_transform), а до тестових застосовуються вже навчені
   параметри (transform). Це запобігає витоку даних (data leakage).

3. Різні solvers (lbfgs, liblinear, saga) дають подібну точність на цьому датасеті.
   lbfgs є оптимальним вибором для задач середнього розміру.

4. Нова модель з ознаками Year/Month із дати та коректною нормалізацією
   демонструє покращення порівняно з базовою завдяки більш інформативним ознакам
   та правильному розподілу train/test без витоку даних у часі.
""")


Висновки:
1. Розбиття на тренувальну/тестову вибірки за часовим принципом (останній рік —
   тестова) дозволяє змоделювати реальні умови: навчання на "минулому",
   оцінка на умовному "майбутньому". Це запобігає надмірному оптимізму в метриках.

2. Трансформери (SimpleImputer, StandardScaler, OneHotEncoder) навчаються лише на
   тренувальних даних (fit_transform), а до тестових застосовуються вже навчені
   параметри (transform). Це запобігає витоку даних (data leakage).

3. Різні solvers (lbfgs, liblinear, saga) дають подібну точність на цьому датасеті.
   lbfgs є оптимальним вибором для задач середнього розміру.

4. Нова модель з ознаками Year/Month із дати та коректною нормалізацією
   демонструє покращення порівняно з базовою завдяки більш інформативним ознакам
   та правильному розподілу train/test без витоку даних у часі.

